# Enhanced LLM-Aided Testbench Generation Assignment

This notebook is adapted for the homework submission. It runs **two tutorial examples** and **one custom module**, then shows **artifact inspection**, **simulation output**, and a **bug-detection demo**.

## Section 1: Installation and Setup

First, we install the required dependencies and set up the environment. The only external dependency is the OpenAI API client.

In [ ]:
# Install required packages
!pip install openai
!apt-get install iverilog
!git clone https://github.com/FCHXWH823/LLM-aided-Testbench-Generation.git

# Import standard libraries
import os
import json
import sys
import subprocess
from typing import Dict, Any, List, Optional
import re

print("✓ Dependencies installed successfully")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Suggested packages:
  gtkwave
The following NEW packages will be installed:
  iverilog
0 upgraded, 1 newly installed, 0 to remove and 2 not upgraded.
Need to get 2,130 kB of archives.
After this operation, 6,749 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 iverilog amd64 11.0-1.1 [2,130 kB]
Fetched 2,130 kB in 1s (1,788 kB/s)
Selecting previously unselected package iverilog.
(Reading database ... 117540 files and directories currently installed.)
Preparing to unpack .../iverilog_11.0-1.1_amd64.deb ...
Unpacking iverilog (11.0-1.1) ...
Setting up iverilog (11.0-1.1) ...
Processing triggers for man-db (2.10.2-1) ...
Cloning into 'LLM-aided-Testbench-Generation'...
remote: Enumerating objects: 134, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (92/92), done.
remote: Total 134 (delta 57), reused 1

## Section 2: API Key Configuration

Insert your API key in the code cell below before running the notebook. Do **not** commit the key to GitHub.

In [ ]:
# Set your OpenAI API key here (or use environment variable)
os.environ['OPENAI_API_KEY'] = ''

# Check if API key is set
if os.environ['OPENAI_API_KEY']:
    print("✓ OpenAI API key is configured")
else:
    print("⚠ Warning: OpenAI API key not set. Running in demo mode.")
    print("  Set your API key with: os.environ['OPENAI_API_KEY'] = 'your-key'")

✓ OpenAI API key is configured


## Section 3: Create Homework Output Directory

In [ ]:
# Create output directory for the homework submission
import os
output_dir = "HW_TestbenchGen"
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory ready: {os.path.abspath(output_dir)}")

Output directory ready: /content/HW_TestbenchGen


## Section 4: LLM Client Implementation

The `LLMClient` class handles all interactions with the OpenAI API. It provides a unified interface for generating text using GPT models.

In [ ]:
"""
LLM Client for interacting with language models.
Supports multiple LLM providers (OpenAI, Anthropic, etc.)
"""

import os
import json
from typing import Optional, Dict, Any
from xml.parsers.expat import model
import openai


class LLMClient:
    """Client for interacting with LLM APIs."""

    def __init__(self, api_key: Optional[str] = None, model: str = "gpt-4", provider: str = "openai"):
        """
        Initialize LLM client.

        Args:
            api_key: API key for the LLM provider (if None, reads from environment)
            model: Model name to use
            provider: LLM provider ('openai', 'anthropic', etc.)
        """
        self.provider = provider
        self.model = model
        self.api_key = api_key or os.getenv("OPENAI_API_KEY")
        self.client = openai.OpenAI(api_key=api_key)

    def generate(self, prompt: str, system_prompt: Optional[str] = None,
                 temperature: float = 0.7, max_tokens: int = 4000) -> str:
        """
        Generate text using the LLM.

        Args:
            prompt: User prompt
            system_prompt: System prompt for the model
            temperature: Sampling temperature (0-1)
            max_tokens: Maximum tokens to generate

        Returns:
            Generated text response
        """
        try:
            if self.provider == "openai":
                messages = []
                if system_prompt:
                    messages.append({"role": "system", "content": system_prompt})
                messages.append({"role": "user", "content": prompt})

                response = self.client.chat.completions.create(
                    model=self.model,
                    messages=messages,
                    temperature=temperature,
                    max_tokens=max_tokens,
                )
                return response.choices[0].message.content
            else:
                raise ValueError(f"Unsupported provider: {self.provider}")
        except Exception as e:
            print(f"Error generating response: {e}")
            return f"Error: {str(e)}"

    def is_available(self) -> bool:
        """Check if the LLM client is properly configured."""
        return self.api_key is not None and len(self.api_key) > 0


## Section 5: Testbench Generator (Step 3)

The `TestbenchGenerator` class generates Verilog testbenches with comprehensive test patterns. It:
- Extracts module information (name, inputs, outputs) from Verilog code
- Uses LLM to generate comprehensive test patterns covering corner cases, boundary values, and random values
- Creates a testbench skeleton without expected outputs (those come in Step 4)

In [ ]:
"""
Step 3: Generate testbench with test patterns (without golden outputs).
"""

from typing import Dict, Any, List


class TestbenchGenerator:
    """Generate Verilog testbench with test patterns using LLM."""

    def __init__(self, llm_client: LLMClient):
        """
        Initialize testbench generator.

        Args:
            llm_client: LLM client instance
        """
        self.llm_client = llm_client

    def generate_testbench(self, description: str, verilog_code: str) -> Dict[str, Any]:
        """
        Generate testbench with comprehensive test patterns.

        Args:
            description: Natural language description of the Verilog module
            verilog_code: Verilog code to be tested

        Returns:
            Dictionary containing:
                - testbench_code: Verilog testbench code (without expected outputs)
                - test_patterns: List of test input patterns
                - module_info: Information about module (name, inputs, outputs)
        """
        # First, extract module information
        module_info = self._extract_module_info(verilog_code)

        # Generate comprehensive test patterns
        system_prompt = """You are an expert in Verilog testbench generation.
Your task is to generate comprehensive test patterns for a given Verilog module.
Generate test patterns that cover:
1. All corner cases
2. Boundary values
3. Typical use cases
4. Edge cases
5. Random values for thorough testing"""

        user_prompt = f"""Given the following Verilog module and its natural language description,
generate a comprehensive Verilog testbench that includes ALL possible test patterns.

Natural Language Description:
{description}

Verilog Module Code:
{verilog_code}

Generate a Verilog testbench that:
1. Declares all necessary signals
2. Instantiates the module under test
3. Includes a systematic set of test patterns covering all cases
4. Uses $display to show inputs for each test (all $display statements are in the initial block and before $finish statement.)
5. Does NOT include expected outputs or assertions yet (we will add those later)
6. Numbers each test case

Please provide:
1. The complete testbench code
2. A list of test patterns in JSON format with test number and input values

Format your response as:
TESTBENCH_CODE:
```verilog
[testbench code here]
```

TEST_PATTERNS (a list of dictionaries. Each dictionary contains only input signal names mapped to their values as plain binary strings (no prefixes like 0b, no spaces). Do not include any test_number field):
```json
[array of test patterns]
```
"""

        response = self.llm_client.generate(user_prompt, system_prompt, max_tokens=4000)

        # Parse the response
        testbench_code = self._extract_section(response, "TESTBENCH_CODE:", "```verilog", "```")
        test_patterns_json = self._extract_section(response, "TEST_PATTERNS:", "```json", "```")

        try:
            test_patterns = eval(test_patterns_json) if test_patterns_json else []
        except:
            test_patterns = []
            print("Warning: Could not parse test patterns JSON")

        return {
            "testbench_code": testbench_code,
            "test_patterns": test_patterns,
            "module_info": module_info,
            "raw_response": response
        }

    def _extract_module_info(self, verilog_code: str) -> Dict[str, Any]:
        """
        Extract module information (name, inputs, outputs) from Verilog code.

        Args:
            verilog_code: Verilog module code

        Returns:
            Dictionary with module information
        """
        lines = verilog_code.strip().split('\n')
        module_name = ""
        inputs = []
        outputs = []

        for line in lines:
            line = line.strip()
            if line.startswith('module'):
                # Extract module name
                parts = line.split()
                if len(parts) >= 2:
                    module_name = parts[1].split('(')[0]
            elif 'input' in line:
                # Extract input signals
                input_part = line.replace('input', '').replace(';', '').replace(',', '').strip()
                if input_part:
                    inputs.append(input_part)
            elif 'output' in line:
                # Extract output signals
                output_part = line.replace('output', '').replace(';', '').replace(',', '').strip()
                if output_part:
                    outputs.append(output_part)

        return {
            "module_name": module_name,
            "inputs": inputs,
            "outputs": outputs
        }

    def _extract_section(self, text: str, marker: str, start_delim: str, end_delim: str) -> str:
        """
        Extract a section from the LLM response between delimiters.

        Args:
            text: Full response text
            marker: Section marker to find
            start_delim: Start delimiter (e.g., "```verilog")
            end_delim: End delimiter (e.g., "```")

        Returns:
            Extracted section content
        """
        try:
            # Find the marker
            marker_idx = text.find(marker)
            if marker_idx == -1:
                return ""

            # Find the start delimiter after the marker
            start_idx = text.find(start_delim, marker_idx)
            if start_idx == -1:
                return ""
            start_idx += len(start_delim)

            # Find the end delimiter
            end_idx = text.find(end_delim, start_idx)
            if end_idx == -1:
                return ""

            return text[start_idx:end_idx].strip()
        except Exception as e:
            print(f"Error extracting section: {e}")
            return ""


## Section 6: Golden Model Generator (Step 4)

The `GoldenModelGenerator` class creates a Python reference implementation from the natural language description. It:
- Converts the description into executable Python code
- Runs all test patterns through the Python model
- Captures expected outputs for verification

In [ ]:
"""
Step 4: Generate Python golden model and compute golden outputs.
"""

import sys
import io
import json
from typing import Dict, Any, List


class GoldenModelGenerator:
    """Generate Python golden model and compute expected outputs."""

    def __init__(self, llm_client: LLMClient):
        """
        Initialize golden model generator.

        Args:
            llm_client: LLM client instance
        """
        self.llm_client = llm_client

    def generate_python_model(self, description: str, module_info: Dict[str, Any]) -> str:
        """
        Generate Python implementation based on natural language description.

        Args:
            description: Natural language description of the module
            module_info: Module information (name, inputs, outputs)

        Returns:
            Python code implementing the module functionality
        """
        system_prompt = """You are an expert in hardware design and Python programming.
Your task is to create a Python function that implements the exact same functionality
as described in the natural language specification."""

        user_prompt = f"""Given the following natural language description of a hardware module,
create a Python function that implements this functionality.

Natural Language Description:
{description}

Module Information:
- Module Name: {module_info.get('module_name', 'unknown')}
- Inputs: {module_info.get('inputs', [])}
- Outputs: {module_info.get('outputs', [])}

Create a Python function named '{module_info.get('module_name', 'module')}_golden' that:
1. Takes the input signals as parameters
2. Computes and returns the output signals
3. Implements the exact functionality described
4. Handles all edge cases properly
5. Returns outputs as a dictionary with output signal names as keys

Provide ONLY the Python function code, no explanations.
Start with 'def {module_info.get('module_name', 'module')}_golden(' and include complete implementation.
"""

        response = self.llm_client.generate(user_prompt, system_prompt, temperature=0.2, max_tokens=3000)

        # Extract Python code
        python_code = self._extract_python_code(response)

        return python_code

    def compute_golden_outputs(self, python_code: str, test_patterns: List[Dict[str, Any]],
                               module_info: Dict[str, Any]) -> List[Dict[str, Any]]:
        """
        Execute the Python golden model with test patterns to get expected outputs.

        Args:
            python_code: Python golden model code
            test_patterns: List of test input patterns
            module_info: Module information

        Returns:
            List of test patterns with golden outputs added
        """
        results = []

        # Execute the Python code in a safe namespace
        namespace = {}
        try:
            exec(python_code, namespace)
        except Exception as e:
            print(f"Error executing Python code: {e}")
            return results

        # Find the golden function
        function_name = f"{module_info.get('module_name', 'module')}_golden"
        golden_func = namespace.get(function_name)

        if not golden_func:
            print(f"Error: Could not find function {function_name}")
            return results

        # Run each test pattern through the golden model
        for pattern in test_patterns:
            try:
                # Extract input values from the pattern
                inputs = pattern.get('inputs', pattern)

                # Call the golden function
                if isinstance(inputs, dict):
                    outputs = golden_func(**inputs)
                else:
                    # If inputs is not a dict, try to call with positional args
                    outputs = golden_func(*inputs.values()) if hasattr(inputs, 'values') else golden_func(inputs)

                # Add outputs to the pattern
                result = pattern.copy()
                result['expected_outputs'] = outputs
                results.append(result)
            except Exception as e:
                print(f"Error computing golden output for pattern {pattern}: {e}")
                result = pattern.copy()
                result['expected_outputs'] = None
                result['error'] = str(e)
                results.append(result)

        return results

    def _extract_python_code(self, text: str) -> str:
        """
        Extract Python code from LLM response.

        Args:
            text: LLM response text

        Returns:
            Extracted Python code
        """
        # Try to find code between ```python and ```
        if "```python" in text:
            start = text.find("```python") + len("```python")
            end = text.find("```", start)
            if end != -1:
                return text[start:end].strip()

        # Try to find code between ``` and ```
        if "```" in text:
            parts = text.split("```")
            if len(parts) >= 3:
                return parts[1].strip()

        # If no code blocks found, look for def statement
        if "def " in text:
            lines = text.split('\n')
            code_lines = []
            in_function = False
            for line in lines:
                if line.strip().startswith('def '):
                    in_function = True
                if in_function:
                    code_lines.append(line)
            return '\n'.join(code_lines)

        return text.strip()


## Section 7: Testbench Updater (Step 5)

The `TestbenchUpdater` class enhances the testbench with verification logic. It:
- Injects expected outputs from the golden model
- Adds pass/fail checking for each test case
- Implements test summary reporting
- Maintains proper Verilog formatting

In [ ]:
"""
Step 5: Update testbench with golden outputs.
"""

from typing import Dict, Any, List
import re
import json


class TestbenchUpdater:
    """Update generated testbench with golden outputs using LLM."""

    def __init__(self, llm_client: LLMClient):
        """
        Initialize testbench updater.

        Args:
            llm_client: LLM client instance
        """
        self.llm_client = llm_client

    def update_testbench(self, testbench_code: str, test_patterns_with_outputs: List[Dict[str, Any]],
                        module_info: Dict[str, Any]) -> str:
        """
        Update testbench code to include expected outputs and verification.

        Args:
            testbench_code: Original testbench code without expected outputs
            test_patterns_with_outputs: Test patterns with golden outputs
            module_info: Module information

        Returns:
            Updated testbench code with assertions and expected outputs
        """
        # Use LLM if available, otherwise fall back to rule-based approach
        if self.llm_client.is_available():
            updated_code = self._llm_update_testbench(testbench_code, test_patterns_with_outputs, module_info)
        else:
            # Fallback to rule-based approach
            updated_code = self._add_verification_logic(testbench_code, test_patterns_with_outputs, module_info)

        return updated_code

    def _llm_update_testbench(self, testbench_code: str, test_patterns_with_outputs: List[Dict[str, Any]],
                             module_info: Dict[str, Any]) -> str:
        """
        Use LLM to update testbench with verification logic.

        Args:
            testbench_code: Original testbench code
            test_patterns_with_outputs: Test patterns with golden outputs
            module_info: Module information

        Returns:
            Updated testbench code with verification logic
        """
        system_prompt = """You are an expert in Verilog testbench development and verification.
Your task is to update a testbench by adding comprehensive verification logic and expected output checks.
You should:
1. Add verification for each test case using the provided expected outputs
2. Track passed and failed test counts
3. Display clear pass/fail messages for each output signal
4. Generate a comprehensive test summary at the end
5. Maintain the original testbench structure and style
6. Use proper Verilog syntax and best practices"""

        # Prepare test patterns data for the LLM
        patterns_str = json.dumps(test_patterns_with_outputs, indent=2)

        user_prompt = f"""Given the following Verilog testbench and test patterns with expected outputs,
update the testbench to include verification logic that checks the actual outputs against the expected outputs.

Original Testbench Code:
```verilog
{testbench_code}
```

Module Information:
- Module Name: {module_info.get('module_name', 'unknown')}
- Inputs: {module_info.get('inputs', [])}
- Outputs: {module_info.get('outputs', [])}

Test Patterns with Expected Outputs:
```json
{patterns_str}
```

Please update the testbench to:
1. Add integer variables 'passed_tests' and 'failed_tests' at the beginning of the initial block (initialized to 0)
2. After each test case (identified by $display statements), add a delay (#10) for outputs to settle
3. For each output signal, compare the actual value against the expected value from the test patterns
4. Display "✓" for passing checks and "✗" for failing checks with actual and expected values
5. Increment passed_tests for each passing check and failed_tests for each failing check
6. At the end of the initial block (before 'end'), add a test summary showing:
   - Total tests run
   - Number passed
   - Number failed
7. Preserve all original test case displays and structure
8. Use proper indentation and formatting

Provide ONLY the complete updated testbench code, no explanations.
Format your response as:
```verilog
[updated testbench code here]
```
"""

        response = self.llm_client.generate(user_prompt, system_prompt, max_tokens=8000)

        # Extract the Verilog code from the response
        updated_code = self._extract_verilog_code(response)

        # If extraction failed, fall back to original with rule-based update
        if not updated_code or len(updated_code) < len(testbench_code) // 2:
            print("Warning: LLM response extraction failed, using rule-based approach")
            updated_code = self._add_verification_logic(testbench_code, test_patterns_with_outputs, module_info)

        return updated_code

    def _extract_verilog_code(self, text: str) -> str:
        """
        Extract Verilog code from LLM response.

        Args:
            text: LLM response text

        Returns:
            Extracted Verilog code
        """
        # Try to find code between ```verilog and ```
        if "```verilog" in text:
            start = text.find("```verilog") + len("```verilog")
            end = text.find("```", start)
            if end != -1:
                return text[start:end].strip()

        # Try to find code between ``` and ```
        if "```" in text:
            parts = text.split("```")
            if len(parts) >= 3:
                # Get the first code block
                code = parts[1].strip()
                # If it starts with a language identifier, remove it
                if code.startswith("verilog\n") or code.startswith("verilog "):
                    code = code.split('\n', 1)[1] if '\n' in code else code
                return code.strip()

        # If no code blocks found, look for module or testbench keywords
        if "module " in text or "initial " in text:
            return text.strip()

        return ""

    def _add_verification_logic(self, testbench_code: str, test_patterns: List[Dict[str, Any]],
                               module_info: Dict[str, Any]) -> str:
        """
        Add verification logic to the testbench.

        Args:
            testbench_code: Original testbench code
            test_patterns: Test patterns with expected outputs
            module_info: Module information

        Returns:
            Testbench code with verification logic added
        """
        lines = testbench_code.split('\n')
        updated_lines = []

        # Track if we're in the initial block
        in_initial = False
        indent_level = 0
        test_case_num = 0

        for i, line in enumerate(lines):
            stripped = line.strip()

            # Detect initial block
            if 'initial' in stripped and 'begin' in stripped:
                in_initial = True
                updated_lines.append(line)
                # Add test result tracking variables after initial begin
                updated_lines.append("    integer passed_tests = 0;")
                updated_lines.append("    integer failed_tests = 0;")
                updated_lines.append("")
                continue

            # Check for test case markers (e.g., $display for test cases)
            if in_initial and '$display' in stripped and ('Test' in stripped or 'test' in stripped):
                # This is likely a test case display
                updated_lines.append(line)

                # Add delay to let outputs settle
                indent = len(line) - len(line.lstrip())
                indent_str = ' ' * indent
                updated_lines.append(f"{indent_str}#10; // Wait for outputs to settle")

                # Add verification for this test case if we have expected outputs
                if test_case_num < len(test_patterns):
                    pattern = test_patterns[test_case_num]
                    if 'expected_outputs' in pattern and pattern['expected_outputs']:
                        verification_lines = self._generate_verification(
                            pattern, module_info, indent
                        )
                        updated_lines.extend(verification_lines)
                    test_case_num += 1
                continue

            # Check for end of initial block
            if in_initial and (stripped == 'end' or stripped.startswith('end')):
                # Add final summary before the end
                indent = len(line) - len(line.lstrip())
                indent_str = ' ' * indent
                updated_lines.append("")
                updated_lines.append(f"{indent_str}// Test Summary")
                updated_lines.append(f'{indent_str}$display("\\n========== Test Summary ==========");')
                updated_lines.append(f'{indent_str}$display("Total Tests: %0d", passed_tests + failed_tests);')
                updated_lines.append(f'{indent_str}$display("Passed: %0d", passed_tests);')
                updated_lines.append(f'{indent_str}$display("Failed: %0d", failed_tests);')
                updated_lines.append(f'{indent_str}$display("==================================\\n");')
                updated_lines.append("")
                updated_lines.append(line)
                in_initial = False
                continue

            updated_lines.append(line)

        return '\n'.join(updated_lines)

    def _generate_verification(self, pattern: Dict[str, Any], module_info: Dict[str, Any],
                              indent: int) -> List[str]:
        """
        Generate verification code for a single test pattern.

        Args:
            pattern: Test pattern with expected outputs
            expected_outputs: Expected output values
            module_info: Module information
            indent: Indentation level

        Returns:
            List of verification code lines
        """
        lines = []
        indent_str = ' ' * indent

        expected = pattern.get('expected_outputs', {})
        if not expected:
            return lines

        # Get output signals
        outputs = module_info.get('outputs', [])

        # Generate verification for each output
        for output in outputs:
            output_name = output.split('[')[0].strip()  # Remove bit width if present
            output_name = output_name.split()[-1]  # Get the signal name

            if output_name in expected:
                expected_value = expected[output_name]

                # Check if expected value is boolean
                if isinstance(expected_value, bool):
                    expected_value = 1 if expected_value else 0

                # Generate comparison
                lines.append(f"{indent_str}if ({output_name} === {expected_value}) begin")
                lines.append(f'{indent_str}    $display("  ✓ {output_name} = %b (expected: {expected_value})", {output_name});')
                lines.append(f"{indent_str}    passed_tests = passed_tests + 1;")
                lines.append(f"{indent_str}end else begin")
                lines.append(f'{indent_str}    $display("  ✗ {output_name} = %b (expected: {expected_value})", {output_name});')
                lines.append(f"{indent_str}    failed_tests = failed_tests + 1;")
                lines.append(f"{indent_str}end")

        return lines

## Section 8: Testbench Generation Pipeline

The `TestbenchPipeline` class coordinates all steps in the generation process. It:
- Manages the flow from description to final testbench
- Handles file I/O for all artifacts
- Provides progress reporting
- Implements graceful degradation when LLM is unavailable

In [ ]:
"""
Main pipeline orchestrating the entire testbench generation process.
"""

import json
import os
from typing import Dict, Any, Optional
import subprocess

class TestbenchPipeline:
    """
    Main pipeline for LLM-aided testbench generation.

    Steps:
    1. Accept natural language description and Verilog code
    2. Generate testbench with test patterns (no golden outputs)
    3. Generate Python golden model from description
    4. Execute golden model with test patterns to get expected outputs
    5. Update testbench with golden outputs and verification logic
    """

    def __init__(self, api_key: Optional[str] = None, model: str = "gpt-4", provider: str = "openai"):
        """
        Initialize the pipeline.

        Args:
            api_key: API key for LLM provider
            model: Model name to use
            provider: LLM provider name
        """
        self.llm_client = LLMClient(api_key, model, provider)
        self.testbench_gen = TestbenchGenerator(self.llm_client)
        self.golden_gen = GoldenModelGenerator(self.llm_client)
        self.testbench_updater = TestbenchUpdater(self.llm_client)

    def run(self, description: str, verilog_code: str, output_dir: str = "output") -> Dict[str, Any]:
        """
        Run the complete testbench generation pipeline.

        Args:
            description: Natural language description of the module
            verilog_code: Verilog code to be tested
            output_dir: Directory to save output files

        Returns:
            Dictionary containing all generated artifacts
        """
        print("=" * 80)
        print("LLM-Aided Testbench Generation Pipeline")
        print("=" * 80)

        # Create output directory
        os.makedirs(output_dir, exist_ok=True)

        # Step 1 & 2: Input handling (description and verilog code are already provided)
        print("\n[Step 1-2] Input: Natural language description and Verilog code received")
        print(f"Description length: {len(description)} characters")
        print(f"Verilog code length: {len(verilog_code)} characters")

        # Step 3: Generate testbench with test patterns
        print("\n[Step 3] Generating testbench with test patterns...")
        if not self.llm_client.is_available():
            print("WARNING: LLM client not configured. Using mock generation.")
            testbench_result = self._mock_testbench_generation(verilog_code)
        else:
            testbench_result = self.testbench_gen.generate_testbench(description, verilog_code)

        print(f"  - Generated testbench with {len(testbench_result['test_patterns'])} test patterns")
        print(f"  - Module: {testbench_result['module_info']['module_name']}")

        # Save initial testbench (without golden outputs)
        initial_tb_path = os.path.join(output_dir, "testbench_initial.v")
        with open(initial_tb_path, 'w') as f:
            f.write(testbench_result['testbench_code'])
        print(f"  - Saved initial testbench to: {initial_tb_path}")

        # Step 4: Generate Python golden model and compute golden outputs
        print("\n[Step 4] Generating Python golden model and computing expected outputs...")
        if not self.llm_client.is_available():
            print("WARNING: LLM client not configured. Using mock golden model.")
            python_code = self._mock_python_model(testbench_result['module_info'])
        else:
            python_code = self.golden_gen.generate_python_model(
                description,
                testbench_result['module_info']
            )

        print(f"  - Generated Python golden model ({len(python_code)} characters)")

        # change testbench_result['test_patterns'] value to integral values
        patterns = []
        for pattern in testbench_result['test_patterns']:
            for key in pattern:
                if isinstance(pattern[key], str):
                    pattern[key] = int(pattern[key], 2)
            patterns.append(pattern)
        testbench_result['test_patterns'] = patterns

        # Save Python golden model
        python_path = os.path.join(output_dir, "golden_model.py")
        with open(python_path, 'w') as f:
            f.write(python_code)
        print(f"  - Saved Python golden model to: {python_path}")

        # Compute golden outputs
        print("  - Computing golden outputs for all test patterns...")
        test_patterns_with_outputs = self.golden_gen.compute_golden_outputs(
            python_code,
            testbench_result['test_patterns'],
            testbench_result['module_info']
        )

        successful_patterns = sum(1 for p in test_patterns_with_outputs
                                 if 'expected_outputs' in p and p['expected_outputs'] is not None)
        print(f"  - Successfully computed outputs for {successful_patterns}/{len(test_patterns_with_outputs)} patterns")

        # Save test patterns with golden outputs
        patterns_path = os.path.join(output_dir, "test_patterns_with_golden.json")
        with open(patterns_path, 'w') as f:
            json.dump(test_patterns_with_outputs, f, indent=2)
        print(f"  - Saved test patterns with golden outputs to: {patterns_path}")

        # Step 5: Update testbench with golden outputs
        print("\n[Step 5] Updating testbench with golden outputs and verification logic...")
        final_testbench = self.testbench_updater.update_testbench(
            testbench_result['testbench_code'],
            test_patterns_with_outputs,
            testbench_result['module_info']
        )

        # Save final testbench
        final_tb_path = os.path.join(output_dir, "testbench_final.v")
        with open(final_tb_path, 'w') as f:
            f.write(final_testbench)
        print(f"  - Saved final testbench to: {final_tb_path}")

        print("\n" + "=" * 80)
        print("Pipeline completed successfully!")
        print("=" * 80)
        print(f"\nGenerated files in '{output_dir}':")
        print(f"  - testbench_initial.v    : Initial testbench with test patterns")
        print(f"  - golden_model.py        : Python reference implementation")
        print(f"  - test_patterns_with_golden.json : Test patterns with expected outputs")
        print(f"  - testbench_final.v      : Final testbench with verification")
        print()

        return {
            'description': description,
            'verilog_code': verilog_code,
            'module_info': testbench_result['module_info'],
            'test_patterns': testbench_result['test_patterns'],
            'initial_testbench': testbench_result['testbench_code'],
            'python_golden_model': python_code,
            'test_patterns_with_outputs': test_patterns_with_outputs,
            'final_testbench': final_testbench,
            'output_dir': output_dir
        }

    def _mock_testbench_generation(self, verilog_code: str) -> Dict[str, Any]:
        """Mock testbench generation when LLM is not available."""
        return {
            'testbench_code': '// Mock testbench - LLM not configured\n' + verilog_code,
            'test_patterns': [],
            'module_info': {
                'module_name': 'unknown',
                'inputs': [],
                'outputs': []
            }
        }

    def _mock_python_model(self, module_info: Dict[str, Any]) -> str:
        """Mock Python model generation when LLM is not available."""
        return f"# Mock Python model - LLM not configured\ndef {module_info['module_name']}_golden():\n    pass\n"

## Section 9: Helper Utilities for Artifact Review and Simulation

In [ ]:

import os
import json
import re
import difflib
import subprocess
from pathlib import Path

def list_generated_files(run_dir):
    print(f"Generated files in {run_dir}:")
    print("=" * 80)
    path = Path(run_dir)
    if not path.exists():
        print("Directory not found.")
    else:
        for fp in sorted(path.iterdir()):
            if fp.is_file():
                print(f"- {fp.name:35s} ({fp.stat().st_size:,} bytes)")
    print("=" * 80)

def print_file(path, max_lines=None):
    path = Path(path)
    if not path.exists():
        print(f"File not found: {path}")
        return
    print(f"Viewing: {path}")
    print("=" * 80)
    with open(path, "r") as f:
        lines = f.readlines()
    shown = lines if max_lines is None else lines[:max_lines]
    for i, line in enumerate(shown, 1):
        print(f"{i:3d}: {line}", end="")
    if max_lines is not None and len(lines) > max_lines:
        print("\n... (truncated)")
    print("=" * 80)

def print_checking_snippet(path):
    path = Path(path)
    if not path.exists():
        print(f"File not found: {path}")
        return
    with open(path, "r") as f:
        lines = f.readlines()
    keywords = re.compile(r"expected|FAIL|PASS|mismatch|assert|!==|===|if\s*\(", re.IGNORECASE)
    selected = [(i + 1, line.rstrip("\n")) for i, line in enumerate(lines) if keywords.search(line)]
    print(f"Key verification lines from {path}:")
    print("=" * 80)
    if not selected:
        print("No obvious checking lines found.")
    else:
        for line_no, line in selected[:40]:
            print(f"{line_no:3d}: {line}")
    print("=" * 80)

def summarize_patterns(path, max_patterns=5):
    path = Path(path)
    if not path.exists():
        print(f"File not found: {path}")
        return
    with open(path, "r") as f:
        patterns = json.load(f)
    print(f"Pattern file: {path}")
    print(f"Total patterns: {len(patterns)}")
    print("=" * 80)
    for i, pattern in enumerate(patterns[:max_patterns], 1):
        print(f"Pattern {i}:")
        print(json.dumps(pattern, indent=2))
    if len(patterns) > max_patterns:
        print(f"... showing first {max_patterns} of {len(patterns)} patterns")
    print("=" * 80)

def run_iverilog(dut_path, tb_path, sim_out):
    print("Compile command:")
    compile_cmd = f"iverilog -g2012 -o {sim_out} {dut_path} {tb_path}"
    print(compile_cmd)
    compile_result = subprocess.run(
        compile_cmd, shell=True, capture_output=True, text=True, timeout=60
    )
    print("=" * 80)
    if compile_result.returncode != 0:
        print("Compilation failed:")
        print(compile_result.stderr)
        return False
    print("Compilation succeeded.")
    run_cmd = f"vvp {sim_out}"
    print("\nRun command:")
    print(run_cmd)
    sim_result = subprocess.run(
        run_cmd, shell=True, capture_output=True, text=True, timeout=60
    )
    print("=" * 80)
    print("Simulation stdout:")
    print(sim_result.stdout)
    if sim_result.stderr.strip():
        print("\nSimulation stderr:")
        print(sim_result.stderr)
    print("=" * 80)
    return sim_result.returncode == 0

def explain_artifacts(example_dir):
    example_dir = Path(example_dir)
    golden_path = example_dir / "golden_model.py"
    patterns_path = example_dir / "test_patterns_with_golden.json"
    initial_tb = example_dir / "testbench_initial.v"
    final_tb = example_dir / "testbench_final.v"

    print("Artifact explanation summary")
    print("=" * 80)

    if golden_path.exists():
        golden_text = golden_path.read_text()
        print("1) Golden model:")
        print("- This Python file is the reference implementation used to compute expected outputs.")
        print("- It receives the same logical inputs as the DUT test patterns and returns the correct outputs for each pattern.")
        funcs = re.findall(r"def\s+(\w+)\s*\(", golden_text)
        if funcs:
            print(f"- Functions found: {', '.join(funcs)}")
        print()

    if patterns_path.exists():
        with open(patterns_path, "r") as f:
            patterns = json.load(f)
        print("2) Pattern file:")
        print(f"- Stores {len(patterns)} input/output test cases in JSON format.")
        if patterns:
            first = patterns[0]
            print(f"- Example keys in one pattern: {list(first.keys())}")
        print()

    if initial_tb.exists() and final_tb.exists():
        init_lines = initial_tb.read_text().splitlines()
        final_lines = final_tb.read_text().splitlines()
        diff = list(difflib.unified_diff(init_lines, final_lines, fromfile="initial", tofile="final", n=1))
        print("3) Updater effect:")
        if diff:
            print("- The final testbench adds verification logic compared with the initial testbench.")
            for line in diff[:30]:
                print(line)
        else:
            print("- No differences detected.")
    print("=" * 80)


## Section 10: Homework Roadmap

This notebook is organized to match the assignment rubric:

1. **Tutorial Example 1:** 2-to-1 multiplexer  
2. **Tutorial Example 2:** 4-bit adder  
3. **Artifact explanation:** inspect one generated run  
4. **Custom module:** 4-bit priority encoder  
5. **Bug detection:** run the custom module with an intentional bug, then fix it  

Each run writes artifacts into:

- `HW_TestbenchGen/mux/`
- `HW_TestbenchGen/adder/`
- `HW_TestbenchGen/custom/`


## Section 11: Tutorial Example 1 — 2-to-1 Multiplexer

In [ ]:
# Natural language description of the module
mux_description = """
A 2-to-1 multiplexer (MUX).

The module takes two 1-bit input signals (a and b) and one 1-bit select signal (sel).

Functionality:
- Input 'a': First data input (1-bit)
- Input 'b': Second data input (1-bit)
- Input 'sel': Select signal (1-bit)
- Output 'y': Selected output (1-bit)

When sel is 0, the output y should be equal to input a.
When sel is 1, the output y should be equal to input b.

This is a combinational logic circuit with no state or memory.

"""

print("Natural Language Description:")
print("=" * 80)
print(mux_description)
print("=" * 80)

Natural Language Description:

A 2-to-1 multiplexer (MUX).

The module takes two 1-bit input signals (a and b) and one 1-bit select signal (sel).

Functionality:
- Input 'a': First data input (1-bit)
- Input 'b': Second data input (1-bit)
- Input 'sel': Select signal (1-bit)
- Output 'y': Selected output (1-bit)

When sel is 0, the output y should be equal to input a.
When sel is 1, the output y should be equal to input b.

This is a combinational logic circuit with no state or memory.




In [ ]:
# Verilog module to be tested
mux_verilog_code = """
module mux2to1 (
    input wire a,
    input wire b,
    input wire sel,
    output wire y
);
    assign y = sel ? b : a;
endmodule

"""

print("Verilog Module:")
print("=" * 80)
print(mux_verilog_code)
print("=" * 80)

# Save the Verilog code for later simulation
with open(f"{output_dir}/mux2to1.v", "w") as f:
    f.write(mux_verilog_code)

Verilog Module:

module mux2to1 (
    input wire a,
    input wire b,
    input wire sel,
    output wire y
);
    assign y = sel ? b : a;
endmodule




### Run the full pipeline

In [ ]:
# Initialize and run the pipeline
print("\n" + "=" * 80)
print("Running LLM-Aided Testbench Generation Pipeline for MUX")
print("=" * 80 + "\n")

# Create pipeline instance
pipeline = TestbenchPipeline(
    api_key=os.environ.get('OPENAI_API_KEY'),
    model='gpt-4o',
    provider='openai'
)

# Run the complete pipeline
try:
    result = pipeline.run(
        description=mux_description,
        verilog_code=mux_verilog_code,
        output_dir=f"{output_dir}/mux"
    )
    print("\n✓ MUX testbench generation completed successfully!")
    print(f"\nGenerated files are in: {output_dir}/mux/")
except Exception as e:
    print(f"\n✗ Error: {e}")
    import traceback
    traceback.print_exc()


Running LLM-Aided Testbench Generation Pipeline for MUX

LLM-Aided Testbench Generation Pipeline

[Step 1-2] Input: Natural language description and Verilog code received
Description length: 462 characters
Verilog code length: 134 characters

[Step 3] Generating testbench with test patterns...
  - Generated testbench with 8 test patterns
  - Module: mux2to1
  - Saved initial testbench to: HW_TestbenchGen/mux/testbench_initial.v

[Step 4] Generating Python golden model and computing expected outputs...
  - Generated Python golden model (79 characters)
  - Saved Python golden model to: HW_TestbenchGen/mux/golden_model.py
  - Computing golden outputs for all test patterns...
  - Successfully computed outputs for 8/8 patterns
  - Saved test patterns with golden outputs to: HW_TestbenchGen/mux/test_patterns_with_golden.json

[Step 5] Updating testbench with golden outputs and verification logic...
  - Saved final testbench to: HW_TestbenchGen/mux/testbench_final.v

Pipeline completed succe

### Review generated artifacts

In [ ]:
mux_run_dir = f"{output_dir}/mux"
list_generated_files(mux_run_dir)
print_file(f"{mux_run_dir}/testbench_initial.v", max_lines=120)
print_file(f"{mux_run_dir}/golden_model.py", max_lines=200)
summarize_patterns(f"{mux_run_dir}/test_patterns_with_golden.json", max_patterns=5)
print_checking_snippet(f"{mux_run_dir}/testbench_final.v")

Generated files in HW_TestbenchGen/mux:
- golden_model.py                     (79 bytes)
- test_patterns_with_golden.json      (738 bytes)
- testbench_final.v                   (4,040 bytes)
- testbench_initial.v                 (1,389 bytes)
Viewing: HW_TestbenchGen/mux/testbench_initial.v
  1: module tb_mux2to1;
  2:     // Declare signals
  3:     reg a;
  4:     reg b;
  5:     reg sel;
  6:     wire y;
  7: 
  8:     // Instantiate the module under test
  9:     mux2to1 uut (
 10:         .a(a),
 11:         .b(b),
 12:         .sel(sel),
 13:         .y(y)
 14:     );
 15: 
 16:     initial begin
 17:         // Test Case 1
 18:         a = 0; b = 0; sel = 0;
 19:         #10;
 20:         $display("Test 1: a=%b, b=%b, sel=%b, y=%b", a, b, sel, y);
 21: 
 22:         // Test Case 2
 23:         a = 0; b = 0; sel = 1;
 24:         #10;
 25:         $display("Test 2: a=%b, b=%b, sel=%b, y=%b", a, b, sel, y);
 26: 
 27:         // Test Case 3
 28:         a = 0; b = 1; sel = 0;
 29:

### Compile and simulate the final multiplexer testbench

In [ ]:
run_iverilog(
    dut_path=f"{output_dir}/mux2to1.v",
    tb_path=f"{output_dir}/mux/testbench_final.v",
    sim_out=f"{output_dir}/mux/sim.vvp"
)

Compile command:
iverilog -g2012 -o HW_TestbenchGen/mux/sim.vvp HW_TestbenchGen/mux2to1.v HW_TestbenchGen/mux/testbench_final.v
Compilation succeeded.

Run command:
vvp HW_TestbenchGen/mux/sim.vvp
Simulation stdout:
Test 1: a=0, b=0, sel=0, y=0
✓ Test 1 Passed: y=0 (Expected: 0)
Test 2: a=0, b=0, sel=1, y=0
✓ Test 2 Passed: y=0 (Expected: 0)
Test 3: a=0, b=1, sel=0, y=0
✓ Test 3 Passed: y=0 (Expected: 0)
Test 4: a=0, b=1, sel=1, y=1
✓ Test 4 Passed: y=1 (Expected: 1)
Test 5: a=1, b=0, sel=0, y=1
✓ Test 5 Passed: y=1 (Expected: 1)
Test 6: a=1, b=0, sel=1, y=0
✓ Test 6 Passed: y=0 (Expected: 0)
Test 7: a=1, b=1, sel=0, y=1
✓ Test 7 Passed: y=1 (Expected: 1)
Test 8: a=1, b=1, sel=1, y=1
✓ Test 8 Passed: y=1 (Expected: 1)
Test Summary: Total=8, Passed=8, Failed=0



True

## Section 12: Tutorial Example 2 — 4-bit Adder

In [ ]:
# Natural language description of the 4-bit adder
adder_description = """
A simple 4-bit adder module.

The module takes two 4-bit input signals (a and b) and produces a 4-bit sum output and a 1-bit carry output.

Functionality:
- Input 'a': 4-bit unsigned number
- Input 'b': 4-bit unsigned number
- Output 'sum': 4-bit result of a + b (lower 4 bits)
- Output 'carry': 1-bit carry-out flag (set to 1 if result exceeds 15)

The adder performs unsigned addition of the two 4-bit inputs.
If the result is greater than 15 (0xF), the carry output should be set to 1.

"""

print("Natural Language Description:")
print("=" * 80)
print(adder_description)
print("=" * 80)

Natural Language Description:

A simple 4-bit adder module.

The module takes two 4-bit input signals (a and b) and produces a 4-bit sum output and a 1-bit carry output.

Functionality:
- Input 'a': 4-bit unsigned number
- Input 'b': 4-bit unsigned number
- Output 'sum': 4-bit result of a + b (lower 4 bits)
- Output 'carry': 1-bit carry-out flag (set to 1 if result exceeds 15)

The adder performs unsigned addition of the two 4-bit inputs.
If the result is greater than 15 (0xF), the carry output should be set to 1.




In [ ]:
# Verilog module to be tested
adder_verilog_code = """
module adder4bit (
    input wire [3:0] a,
    input wire [3:0] b,
    output wire [3:0] sum,
    output wire carry
);
    wire [4:0] result;
    assign result = a + b;
    assign sum = result[3:0];
    assign carry = result[4];
endmodule

"""

print("Verilog Module:")
print("=" * 80)
print(adder_verilog_code)
print("=" * 80)

# Save the Verilog code
with open(f"{output_dir}/adder4bit.v", "w") as f:
    f.write(adder_verilog_code)

Verilog Module:

module adder4bit (
    input wire [3:0] a,
    input wire [3:0] b,
    output wire [3:0] sum,
    output wire carry
);
    wire [4:0] result;
    assign result = a + b;
    assign sum = result[3:0];
    assign carry = result[4];
endmodule




### Run the full pipeline

In [ ]:
# Run pipeline for the adder
print("\n" + "=" * 80)
print("Running LLM-Aided Testbench Generation Pipeline for 4-bit Adder")
print("=" * 80 + "\n")

try:
    result = pipeline.run(
        description=adder_description,
        verilog_code=adder_verilog_code,
        output_dir=f"{output_dir}/adder"
    )
    print("\n✓ Adder testbench generation completed successfully!")
    print(f"\nGenerated files are in: {output_dir}/adder/")
except Exception as e:
    print(f"\n✗ Error: {e}")
    import traceback
    traceback.print_exc()


Running LLM-Aided Testbench Generation Pipeline for 4-bit Adder

LLM-Aided Testbench Generation Pipeline

[Step 1-2] Input: Natural language description and Verilog code received
Description length: 491 characters
Verilog code length: 241 characters

[Step 3] Generating testbench with test patterns...
  - Generated testbench with 10 test patterns
  - Module: adder4bit
  - Saved initial testbench to: HW_TestbenchGen/adder/testbench_initial.v

[Step 4] Generating Python golden model and computing expected outputs...
  - Generated Python golden model (337 characters)
  - Saved Python golden model to: HW_TestbenchGen/adder/golden_model.py
  - Computing golden outputs for all test patterns...
  - Successfully computed outputs for 10/10 patterns
  - Saved test patterns with golden outputs to: HW_TestbenchGen/adder/test_patterns_with_golden.json

[Step 5] Updating testbench with golden outputs and verification logic...
  - Saved final testbench to: HW_TestbenchGen/adder/testbench_final.v

Pi

### Review generated artifacts

In [ ]:
adder_run_dir = f"{output_dir}/adder"
list_generated_files(adder_run_dir)
print_checking_snippet(f"{adder_run_dir}/testbench_final.v")
summarize_patterns(f"{adder_run_dir}/test_patterns_with_golden.json", max_patterns=5)

Generated files in HW_TestbenchGen/adder:
- golden_model.py                     (337 bytes)
- test_patterns_with_golden.json      (993 bytes)
- testbench_final.v                   (5,098 bytes)
- testbench_initial.v                 (1,561 bytes)
Key verification lines from HW_TestbenchGen/adder/testbench_final.v:
 17:     integer passed_tests = 0;
 18:     integer failed_tests = 0;
 26:         if (sum === 4'b0000 && carry === 1'b0) begin
 27:             $display("  ✓ Test 1 passed");
 28:             passed_tests = passed_tests + 1;
 31:             $display("  ✗ Test 1 failed: Expected sum=0000, carry=0");
 32:             failed_tests = failed_tests + 1;
 38:         if (sum === 4'b0000 && carry === 1'b1) begin
 39:             $display("  ✓ Test 2 passed");
 40:             passed_tests = passed_tests + 1;
 43:             $display("  ✗ Test 2 failed: Expected sum=0000, carry=1");
 44:             failed_tests = failed_tests + 1;
 50:         if (sum === 4'b0000 && carry === 1'b1)

### Compile and simulate the final adder testbench

In [ ]:
run_iverilog(
    dut_path=f"{output_dir}/adder4bit.v",
    tb_path=f"{output_dir}/adder/testbench_final.v",
    sim_out=f"{output_dir}/adder/sim.vvp"
)

Compile command:
iverilog -g2012 -o HW_TestbenchGen/adder/sim.vvp HW_TestbenchGen/adder4bit.v HW_TestbenchGen/adder/testbench_final.v
Compilation succeeded.

Run command:
vvp HW_TestbenchGen/adder/sim.vvp
Simulation stdout:
Test 1: a=0000, b=0000, sum=0000, carry=0
  ✓ Test 1 passed
Test 2: a=1111, b=0001, sum=0000, carry=1
  ✓ Test 2 passed
Test 3: a=1000, b=1000, sum=0000, carry=1
  ✓ Test 3 passed
Test 4: a=1111, b=1111, sum=1110, carry=1
  ✓ Test 4 passed
Test 5: a=0001, b=0001, sum=0010, carry=0
  ✓ Test 5 passed
Test 6: a=1110, b=0001, sum=1111, carry=0
  ✓ Test 6 passed
Test 7: a=0101, b=0011, sum=1000, carry=0
  ✓ Test 7 passed
Test 8: a=0110, b=0101, sum=1011, carry=0
  ✓ Test 8 passed
Test 9: a=1010, b=0101, sum=1111, carry=0
  ✓ Test 9 passed
Test 10: a=0011, b=1100, sum=1111, carry=0
  ✓ Test 10 passed
Test Summary:
  Total tests run:          10
  Number passed:          10
  Number failed:           0



True

## Section 13: Required Artifact Explanation

Run the next cell after the multiplexer pipeline finishes. It prints a short explanation of `golden_model.py`, `test_patterns_with_golden.json`, and the updater’s before/after effect.

In [ ]:
explain_artifacts(f"{output_dir}/mux")

Artifact explanation summary
1) Golden model:
- This Python file is the reference implementation used to compute expected outputs.
- It receives the same logical inputs as the DUT test patterns and returns the correct outputs for each pattern.
- Functions found: mux2to1_golden

2) Pattern file:
- Stores 8 input/output test cases in JSON format.
- Example keys in one pattern: ['a', 'b', 'sel', 'expected_outputs']

3) Updater effect:
- The final testbench adds verification logic compared with the initial testbench.
--- initial

+++ final

@@ -15,3 +15,9 @@

 
+    integer passed_tests;
+    integer failed_tests;
+
     initial begin
+        passed_tests = 0;
+        failed_tests = 0;
+
         // Test Case 1
@@ -20,2 +26,10 @@

         $display("Test 1: a=%b, b=%b, sel=%b, y=%b", a, b, sel, y);
+        #10;
+        if (y === 0) begin
+            $display("✓ Test 1 Passed: y=%b (Expected: 0)", y);
+            passed_tests = passed_tests + 1;
+        end else begin
+            $d

## Section 14: Custom Module — Priority Encoder

In [ ]:
# Natural language description for the custom module
custom_description = """
A 4-bit priority encoder with a valid output.

Inputs:
- req[3:0]: 4-bit request vector

Outputs:
- code[1:0]: binary index of the highest-priority asserted input
- valid: 1 when any request bit is asserted, otherwise 0

Priority rule:
- req[3] has the highest priority
- then req[2]
- then req[1]
- then req[0]

Expected behavior:
- If req is 4'b1000, code must be 2'b11 and valid must be 1
- If req is 4'b0110, code must be 2'b10 and valid must be 1 because bit 2 beats bit 1
- If req is 4'b0001, code must be 2'b00 and valid must be 1
- If req is 4'b0000, valid must be 0 and code should be 2'b00

This is a purely combinational module with no clock and no reset.
Include corner cases and randomized tests.
"""

print("Custom module description:")
print("=" * 80)
print(custom_description)
print("=" * 80)

Custom module description:

A 4-bit priority encoder with a valid output.

Inputs:
- req[3:0]: 4-bit request vector

Outputs:
- code[1:0]: binary index of the highest-priority asserted input
- valid: 1 when any request bit is asserted, otherwise 0

Priority rule:
- req[3] has the highest priority
- then req[2]
- then req[1]
- then req[0]

Expected behavior:
- If req is 4'b1000, code must be 2'b11 and valid must be 1
- If req is 4'b0110, code must be 2'b10 and valid must be 1 because bit 2 beats bit 1
- If req is 4'b0001, code must be 2'b00 and valid must be 1
- If req is 4'b0000, valid must be 0 and code should be 2'b00

This is a purely combinational module with no clock and no reset.
Include corner cases and randomized tests.



### Verilog for the custom module

In [ ]:
custom_verilog_code = """
module priority_encoder4 (
    input  wire [3:0] req,
    output reg  [1:0] code,
    output reg        valid
);
    always @* begin
        valid = |req;
        casex (req)
            4'b1xxx: code = 2'b11;
            4'b01xx: code = 2'b10;
            4'b001x: code = 2'b01;
            4'b0001: code = 2'b00;
            default: code = 2'b00;
        endcase
    end
endmodule
"""

print("Custom Verilog module:")
print("=" * 80)
print(custom_verilog_code)
print("=" * 80)

with open(f"{output_dir}/priority_encoder4.v", "w") as f:
    f.write(custom_verilog_code)

Custom Verilog module:

module priority_encoder4 (
    input  wire [3:0] req,
    output reg  [1:0] code,
    output reg        valid
);
    always @* begin
        valid = |req;
        casex (req)
            4'b1xxx: code = 2'b11;
            4'b01xx: code = 2'b10;
            4'b001x: code = 2'b01;
            4'b0001: code = 2'b00;
            default: code = 2'b00;
        endcase
    end
endmodule



### Run the full pipeline for the custom module

In [ ]:
print("\n" + "=" * 80)
print("Running LLM-Aided Testbench Generation Pipeline for Custom Priority Encoder")
print("=" * 80 + "\n")

try:
    custom_result = pipeline.run(
        description=custom_description,
        verilog_code=custom_verilog_code,
        output_dir=f"{output_dir}/custom"
    )
    print("\n✓ Custom testbench generation completed successfully!")
    print(f"\nGenerated files are in: {output_dir}/custom/")
except Exception as e:
    print(f"\n✗ Error: {e}")
    import traceback
    traceback.print_exc()


Running LLM-Aided Testbench Generation Pipeline for Custom Priority Encoder

LLM-Aided Testbench Generation Pipeline

[Step 1-2] Input: Natural language description and Verilog code received
Description length: 711 characters
Verilog code length: 385 characters

[Step 3] Generating testbench with test patterns...
  - Generated testbench with 10 test patterns
  - Module: priority_encoder4
  - Saved initial testbench to: HW_TestbenchGen/custom/testbench_initial.v

[Step 4] Generating Python golden model and computing expected outputs...
  - Generated Python golden model (458 characters)
  - Saved Python golden model to: HW_TestbenchGen/custom/golden_model.py
  - Computing golden outputs for all test patterns...
  - Successfully computed outputs for 10/10 patterns
  - Saved test patterns with golden outputs to: HW_TestbenchGen/custom/test_patterns_with_golden.json

[Step 5] Updating testbench with golden outputs and verification logic...


### Review generated custom-module artifacts

In [ ]:
custom_run_dir = f"{output_dir}/custom"
list_generated_files(custom_run_dir)
print_checking_snippet(f"{custom_run_dir}/testbench_final.v")
summarize_patterns(f"{custom_run_dir}/test_patterns_with_golden.json", max_patterns=8)

### Compile and simulate the final custom-module testbench

In [ ]:
run_iverilog(
    dut_path=f"{output_dir}/priority_encoder4.v",
    tb_path=f"{output_dir}/custom/testbench_final.v",
    sim_out=f"{output_dir}/custom/sim.vvp"
)

## Section 15: Bug Detection Demo

This section intentionally introduces a bug into the custom DUT, reruns simulation to show a failure, and then restores the correct DUT for a passing run.

In [ ]:
buggy_custom_verilog_code = """
module priority_encoder4 (
    input  wire [3:0] req,
    output reg  [1:0] code,
    output reg        valid
);
    always @* begin
        valid = |req;
        casex (req)
            4'b1xxx: code = 2'b10;  // Intentional bug: should be 2'b11
            4'b01xx: code = 2'b10;
            4'b001x: code = 2'b01;
            4'b0001: code = 2'b00;
            default: code = 2'b00;
        endcase
    end
endmodule
"""

buggy_path = f"{output_dir}/priority_encoder4_buggy.v"
with open(buggy_path, "w") as f:
    f.write(buggy_custom_verilog_code)

print("Buggy DUT written to:", buggy_path)
print(buggy_custom_verilog_code)

### Simulate the buggy DUT against the final generated testbench

In [ ]:
run_iverilog(
    dut_path=f"{output_dir}/priority_encoder4_buggy.v",
    tb_path=f"{output_dir}/custom/testbench_final.v",
    sim_out=f"{output_dir}/custom/sim_buggy.vvp"
)

### Restore the correct DUT and simulate again

In [28]:
with open(f"{output_dir}/priority_encoder4.v", "w") as f:
    f.write(custom_verilog_code)

run_iverilog(
    dut_path=f"{output_dir}/priority_encoder4.v",
    tb_path=f"{output_dir}/custom/testbench_final.v",
    sim_out=f"{output_dir}/custom/sim_fixed.vvp"
)

Compile command:
iverilog -g2012 -o HW_TestbenchGen/custom/sim_fixed.vvp HW_TestbenchGen/priority_encoder4.v HW_TestbenchGen/custom/testbench_final.v
Compilation succeeded.

Run command:
vvp HW_TestbenchGen/custom/sim_fixed.vvp
Simulation stdout:
Test 1: req = 0000
  code: ✓ (expected 00, got 00)
  valid: ✓ (expected 0, got 0)
Test 2: req = 1000
  code: ✓ (expected 11, got 11)
  valid: ✓ (expected 1, got 1)
Test 3: req = 0100
  code: ✓ (expected 10, got 10)
  valid: ✓ (expected 1, got 1)
Test 4: req = 0010
  code: ✓ (expected 01, got 01)
  valid: ✓ (expected 1, got 1)
Test 5: req = 0001
  code: ✓ (expected 00, got 00)
  valid: ✓ (expected 1, got 1)
Test 6: req = 1100
  code: ✓ (expected 11, got 11)
  valid: ✓ (expected 1, got 1)
Test 7: req = 0110
  code: ✓ (expected 10, got 10)
  valid: ✓ (expected 1, got 1)
Test 8: req = 0011
  code: ✓ (expected 01, got 01)
  valid: ✓ (expected 1, got 1)
Test 9: req = 1010
  code: ✓ (expected 11, got 11)
  valid: ✓ (expected 1, got 1)
Test 10: req = 

True

## Section 16: Submission Checklist

Before exporting your submission:

- Run the notebook **top to bottom**
- Verify the outputs are visible
- Keep the final notebook name as **`testbenchgen_hw.ipynb`**
- Keep generated artifacts inside **`HW_TestbenchGen/`**
- Export your separate report as **`TestbenchGen_Report.pdf`**
- Invite **`weihuax6@gmail.com`** as a collaborator on GitHub


## Appendix: View All Generated Files

In [29]:
for root, dirs, files in os.walk(output_dir):
    level = root.replace(output_dir, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for f in sorted(files):
        print(f"{subindent}{f}")

HW_TestbenchGen/
  adder4bit.v
  mux2to1.v
  priority_encoder4.v
  priority_encoder4_buggy.v
  custom/
    golden_model.py
    sim.vvp
    sim_buggy.vvp
    sim_fixed.vvp
    test_patterns_with_golden.json
    testbench_final.v
    testbench_initial.v
  adder/
    golden_model.py
    sim.vvp
    test_patterns_with_golden.json
    testbench_final.v
    testbench_initial.v
  mux/
    golden_model.py
    sim.vvp
    test_patterns_with_golden.json
    testbench_final.v
    testbench_initial.v
